# Proof of concept: feature vector

Can a policy learn to drive one track from the state vector alone? One track, no
domain randomization, no holdout.

**Already answered by the recorded 200-trial study: yes.** 36 trials completed,
the best reaching 100% lap completion and 0% off-track at 44.4 m per episode.
`study.db` under `runs/proof/` holds the run, so re-executing the HPO cell
resumes rather than restarts. The camera half is `proof_camera.ipynb`.

In [ ]:
import inspect
import json
import os

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
from deepracer_genesis.experiment import (
    Evaluation,
    FeatureEnvironment,
    PPO,
    RewardShaping,
    VectorPolicy,
    run,
)
from deepracer_genesis.randomization.spaces import Choice, FloatRange, IntRange
import optuna

from experiments import rewards

In [ ]:
SEED = 0
ROOT = "runs/proof"
TRACK = "reinvent_base"

# Episodes end on the task, not the clock: 3 laps truncates (bootstrapped,
# never penalized). max_laps counts finish-line CROSSINGS, and random_start
# spawns mid-lap, so the distance covered averages 2.5 laps rather than 3.
MAX_LAPS = 3
EPISODE_LENGTH_S = 60.0

# no renderer to feed, so envs are cheap and trials cost ~16 s each
FEATURE_ENVS = 1024

# all eighteen rewards; a trial is cheap enough for real coverage
REWARD_NAMES = None                # None -> every reward in rewards.py

FEATURE_TRIALS = 200
FEATURE_TRIAL_STEPS = 1_000_000
FEATURE_TRIAL_EVAL_EVERY = 250_000

# A draw under this at any eval has collapsed, so kill it. Validated on the
# recorded 200-trial study: all 8 later-eval kills were monotonically declining.
FEATURE_TAKEOFF_MIN_M = 2.0

FEATURE_FINAL_STEPS = 5_000_000
FEATURE_FINAL_EVAL_EVERY = 500_000

os.makedirs(ROOT, exist_ok=True)

In [ ]:
# live curves for every trial and the final run; each run dir is its own TB run
%load_ext tensorboard
%tensorboard --logdir $ROOT

## Rewards

The collection is `experiments/rewards.py`, a torch port of
[DeepRacerRewardFunctionsCollection](https://github.com/Machine-Learning-Labs/DeepRacerRewardFunctionsCollection).
`lap_bonus_baseline` is the control: it pays 1.0 only on a finish-line
crossing, and it reached 100% completion here. `run()` seeds python, numpy and
torch from `spec.seed`, so `SEED` alone determines the run.

In [ ]:
#: every reward defined in experiments/rewards.py, in definition order
REWARDS = {name: fn for name, fn in vars(rewards).items()
           if inspect.isfunction(fn) and fn.__module__ == rewards.__name__}
REWARD_NAMES = REWARD_NAMES or list(REWARDS)

print(f"{len(REWARDS)} rewards available")
print(f"searching {len(REWARD_NAMES)}")

In [ ]:
SEARCH_SPACE = {
    "lr": FloatRange(1e-4, 1e-2, log=True),
    "entropy_coef": FloatRange(1e-3, 3e-2, log=True),
    "epochs": IntRange(3, 8),
    "clip": FloatRange(0.1, 0.3),
    "gamma": Choice([0.98, 0.99, 0.995, 0.999]),
    "minibatches": Choice([2, 4, 8]),
}


def suggest_reward(trial: optuna.Trial) -> str:
    """Samples which reward to train on.

    Args:
        trial: The Optuna trial.

    Returns:
        A REWARDS key.
    """
    return trial.suggest_categorical("reward", REWARD_NAMES)


def suggest_mlp(trial: optuna.Trial) -> tuple:
    """Samples the MLP as a halving funnel of layer widths.

    Args:
        trial: The Optuna trial.

    Returns:
        The ``hidden`` tuple for the policy's ``mlp`` dict.
    """
    depth = trial.suggest_int("depth", 2, 4)
    width = trial.suggest_categorical("width", [128, 256, 512])
    return tuple(max(width // 2 ** i, 32) for i in range(depth))

In [ ]:
def feature_spec(*, lr, entropy_coef, epochs, clip, gamma, minibatches, mlp,
                 reward, steps, eval_every, variant, group="proof_feature"):
    """Builds one feature-vector proof spec.

    Args:
        lr: PPO learning rate.
        entropy_coef: PPO entropy bonus coefficient.
        epochs: PPO optimization epochs per iteration.
        clip: PPO surrogate clipping range.
        gamma: Reward discount factor.
        minibatches: Minibatches per PPO epoch.
        mlp: ``hidden`` tuple for the policy.
        reward: A REWARDS key naming which reward to train on.
        steps: Total env-steps, a spec field rather than a stage argument.
        eval_every: Periodic deterministic-eval cadence in env-steps.
        variant: Run-dir variant tag.
        group: Run-dir group tag.

    Returns:
        The validated frozen ExperimentSpec.
    """
    return (
        FeatureEnvironment(num_envs=FEATURE_ENVS, tracks=(TRACK,),
                           max_laps=MAX_LAPS,
                           episode_length_s=EPISODE_LENGTH_S)
        >> VectorPolicy(keys=("state",),
                        mlp={"hidden": mlp, "activation": "relu"})
        # every reward returns the step reward itself, so scales stay empty
        >> RewardShaping(fn=REWARDS[reward], scales={})
        >> PPO(lr=lr, entropy_coef=entropy_coef, epochs=epochs, clip=clip,
               gamma=gamma, minibatches=minibatches)
        >> Evaluation(charts=True)
    ).build(seed=SEED, total_env_steps=steps, eval_every_steps=eval_every,
            group=group, variant=variant)

## HPO

Trials are ranked on `completion_rate`, ground truth the reward shaping cannot
inflate, with `mean_progress_m` as the tie-break.

Note the recorded study saturates: once completion pins at 1.000 and
`max_laps` caps progress, the top trials tie to four decimal places while their
speeds differ 2x. If you rerun this to pick a *fast* policy rather than a
finishing one, fall through to `lap_time_s` in `score`.

In [ ]:
def make_study(name: str, trial_steps: int, eval_every: int) -> optuna.Study:
    """Creates or resumes one seeded, pruning, sqlite-backed study.

    Args:
        name: Study name; also its row in ``<ROOT>/study.db``.
        trial_steps: Max env-steps per trial, the pruner's max resource.
        eval_every: Eval cadence, the pruner's min resource.

    Returns:
        The Optuna study.
    """
    return optuna.create_study(
        direction="maximize",
        study_name=name,
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.HyperbandPruner(
            min_resource=eval_every, max_resource=trial_steps,
            reduction_factor=3),
        storage=f"sqlite:///{ROOT}/study.db",
        load_if_exists=True,
    )


def stop(trial: optuna.Trial, outcome: str, reason: str):
    """Records why a trial stopped, then prunes it.

    Optuna persists neither a ``TrialPruned`` message nor a caught exception's
    text, so the reason survives only as a user attribute.

    Args:
        trial: The Optuna trial.
        outcome: Machine-readable tag, e.g. "parked" or "oom_predicted".
        reason: Human-readable detail.

    Raises:
        optuna.TrialPruned: Always.
    """
    trial.set_user_attr("outcome", outcome)
    trial.set_user_attr("reason", reason)
    raise optuna.TrialPruned(f"{outcome}: {reason}")


def report_to(trial: optuna.Trial, takeoff_min: float):
    """Builds the periodic-eval callback that stops a trial early.

    Two independent stops. ``mean_progress_m`` is what Hyperband ranks on
    because ``completion_rate`` sits at 0 for most of a trial budget and cannot
    discriminate. The absolute floor kills a draw that has collapsed, which
    Hyperband cannot do without peers to rank it against.

    Args:
        trial: The Optuna trial.
        takeoff_min: Mean progress (m) a live draw must hold at every eval.

    Returns:
        An ``on_eval(frames, metrics)`` callback.

    Raises:
        optuna.TrialPruned: Raised inside the callback, which stops the run.
    """
    def report(frames: int, metrics: dict) -> None:
        progress = metrics["mean_progress_m"]
        trial.report(progress, frames)
        if progress < takeoff_min:
            stop(trial, "parked",
                 f"{progress:.2f} m at {frames:,} steps, floor {takeoff_min} m")
        if trial.should_prune():
            stop(trial, "hyperband",
                 f"ranked below peers at {frames:,} steps ({progress:.2f} m)")

    return report


def score(metrics: dict) -> float:
    """Ranks a finished trial: completion first, progress as the tie-break.

    Args:
        metrics: The final deterministic eval's metrics.

    Returns:
        The objective value to maximize.
    """
    return float(metrics["completion_rate"] * 100.0
                 + 0.05 * metrics["mean_progress_m"])


def leaderboard(study: optuna.Study) -> pd.DataFrame:
    """Tabulates a finished study, best score first.

    Args:
        study: The Optuna study.

    Returns:
        One row per trial.
    """
    return pd.DataFrame([
        {"trial": t.number, "state": t.state.name, "score": t.value,
         "completion": t.user_attrs.get("completion"),
         "progress_m": t.user_attrs.get("progress_m"), **t.params}
        for t in study.trials]).sort_values("score", ascending=False)


def outcomes(study: optuna.Study) -> pd.DataFrame:
    """Tabulates every trial with the recorded reason it stopped.

    Args:
        study: The Optuna study.

    Returns:
        One row per trial: number, state, outcome, reason, score and params.
    """
    return pd.DataFrame([
        {"trial": t.number, "state": t.state.name,
         "outcome": t.user_attrs.get("outcome", "unrecorded"),
         "reason": t.user_attrs.get("reason", ""),
         "score": t.value, **t.params}
        for t in study.trials])


def failure_report(study: optuna.Study, name: str) -> pd.DataFrame:
    """Prints why trials stopped, overall and per reward, and returns the detail.

    A reward whose every trial parked is a finding about that reward; one whose
    trials Hyperband pruned merely lost a ranking.

    Args:
        study: The Optuna study.
        name: Label used in the printed headings.

    Returns:
        The rows for trials that did not complete.
    """
    table = outcomes(study)
    print(f"=== {name}: {len(table)} trials ===")
    print(table["outcome"].value_counts().to_string(), "\n")
    if "reward" in table:
        print((table.groupby("reward")["outcome"]
               .value_counts().unstack(fill_value=0)).to_string(), "\n")
    return table[table["outcome"] != "completed"]


optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
def objective(trial: optuna.Trial) -> float:
    """Trains one sampled feature config and scores it on completion.

    Args:
        trial: The Optuna trial.

    Returns:
        The trial score: completion, with progress as the tie-break.

    Raises:
        optuna.TrialPruned: On a collapsed draw or a Hyperband cut.
    """
    cfg = {name: space.suggest(trial, name)
           for name, space in SEARCH_SPACE.items()}
    cfg["reward"] = suggest_reward(trial)
    cfg["mlp"] = suggest_mlp(trial)
    trial.set_user_attr("cfg", cfg)

    record = run(feature_spec(**cfg, steps=FEATURE_TRIAL_STEPS,
                              eval_every=FEATURE_TRIAL_EVAL_EVERY,
                              group="proof_feature_hpo",
                              variant=f"t{trial.number:03d}"),
                 root=ROOT, on_eval=report_to(trial, FEATURE_TAKEOFF_MIN_M))

    trial.set_user_attr("outcome", "completed")
    trial.set_user_attr("reason", "")
    trial.set_user_attr("completion", record.metrics["completion_rate"])
    trial.set_user_attr("progress_m", record.metrics["mean_progress_m"])
    return score(record.metrics)


study = make_study("proof_feature", FEATURE_TRIAL_STEPS,
                   FEATURE_TRIAL_EVAL_EVERY)
study.optimize(objective, n_trials=FEATURE_TRIALS, catch=(RuntimeError,))

display(leaderboard(study))
BEST = study.best_trial.user_attrs["cfg"]
print("winner:", BEST)

## What did not work

Each trial records an `outcome` tag and a `reason` before it stops, because
Optuna persists neither a `TrialPruned` message nor an exception's text.

In the recorded study: 36 completed, 52 parked, 112 pruned by Hyperband, all 18
reward arms drawn. The twelve rewards that never completed are mostly the
1e5-scale families whose value loss explodes.

In [ ]:
FAILED = failure_report(study, "feature")
display(FAILED[["trial", "outcome", "reason", "reward"]])

## Final training

The winning config, trained long. Feature training has no takeoff lottery, so
one run suffices.

In [ ]:
FINAL = run(feature_spec(**BEST, steps=FEATURE_FINAL_STEPS,
                         eval_every=FEATURE_FINAL_EVAL_EVERY,
                         variant="final"),
            root=ROOT)
print({k: round(v, 3) for k, v in FINAL.metrics.items()
       if isinstance(v, float)})

## Result

`completion_rate` is the claim. The table picks up `proof_camera.ipynb`'s row if
that notebook has run.

In [ ]:
def proof_row(record, cfg: dict) -> dict:
    """Builds this notebook's row of the proof table.

    Args:
        record: The finished run's EvalRecord.
        cfg: The winning config that produced the run.

    Returns:
        A flat dict of the metrics the proof claims.
    """
    return {
        "modality": "feature",
        "reward": cfg["reward"],
        "completion_rate": record.metrics["completion_rate"],
        "mean_laps": record.metrics["mean_laps"],
        "lap_time_s": record.metrics["lap_time_s"],
        "mean_progress_m": record.metrics["mean_progress_m"],
        "offtrack_rate": record.metrics["offtrack_rate"],
        "mean_speed_mps": record.metrics["mean_speed_mps"],
        "env_steps": record.train["total_env_steps"],
        "wall_clock_s": record.train["wall_clock_s"],
        "steps_per_s": record.train["steps_per_s"],
    }


ROW = proof_row(FINAL, BEST)
with open(os.path.join(ROOT, "proof_row.json"), "w") as f:
    json.dump(ROW, f, indent=2)

# the other notebook's row, if it has already run
rows = [ROW]
for other in ("runs/proof/proof_row.json", "runs/proof_camera/proof_row.json"):
    if os.path.exists(other) and other != os.path.join(ROOT, "proof_row.json"):
        rows.append(json.load(open(other)))
display(pd.DataFrame(rows))

fig, ax = plt.subplots(figsize=(7, 4))
history = pd.DataFrame(FINAL.eval_history)
ax.plot(history["frames"], history["completion_rate"], marker="o",
        label="feature")
ax.set_xlabel("env steps")
ax.set_ylabel("completion rate")
ax.set_title(f"Learning to lap {TRACK}")
ax.legend()
fig.tight_layout()